# NumPy: arrays para el análisis de datos

**Análisis de datos con Python · UT2 · Manejo de datos con pandas**

Apuntes principales de NumPy del módulo. Se trabajan en la jornada 6 y son la base de todo lo que viene después: pandas, los gráficos y los modelos están construidos sobre NumPy.

> **Cómo usar este cuaderno**
> - Ejecuta las celdas **en orden**, de arriba abajo. Muchas usan variables de celdas anteriores.
> - Si algo no cuadra, *Kernel → Restart Kernel and Run All Cells…* y vuelve a empezar.
> - El fichero `ventas_tienda_mes.csv` tiene que estar **en la misma carpeta** que este cuaderno (lo usan los apartados 12, 13 y 19).
> - Los ejercicios están en el cuaderno `ejercicios-numpy.ipynb`.

## Índice

1. [Por qué existe el array](#1.-Por-qué-existe-el-array)
2. [Crear arrays](#2.-Crear-arrays)
3. [Atributos y ejes](#3.-Atributos-y-ejes)
4. [Tipos de dato](#4.-Tipos-de-dato)
5. [Índices y rebanadas](#5.-Índices-y-rebanadas)
6. [Máscaras booleanas](#6.-Máscaras-booleanas)
7. [Indexación con listas de índices](#7.-Indexación-con-listas-de-índices)
8. [Vista frente a copia](#8.-Vista-frente-a-copia)
9. [Operaciones vectorizadas](#9.-Operaciones-vectorizadas)
10. [Difusión (broadcasting)](#10.-Difusión-(broadcasting))
11. [Agregación y estadísticos](#11.-Agregación-y-estadísticos)
12. [Agregación por ejes](#12.-Agregación-por-ejes)
13. [Valores ausentes: NaN](#13.-Valores-ausentes:-NaN)
14. [Verificar los resultados](#14.-Verificar-los-resultados)
15. [Leer y guardar datos con NumPy](#15.-Leer-y-guardar-datos-con-NumPy)
16. [Cambiar la forma](#16.-Cambiar-la-forma)
17. [Combinar y dividir arrays](#17.-Combinar-y-dividir-arrays)
18. [Ordenar, valores únicos y conjuntos](#18.-Ordenar,-valores-únicos-y-conjuntos)
19. [Caso práctico: ventas por tienda y mes](#19.-Caso-práctico:-ventas-por-tienda-y-mes)
20. [Errores frecuentes](#20.-Errores-frecuentes)
21. [Chuleta](#21.-Chuleta)

La línea siguiente importa NumPy con el alias **`np`**, que es el que usa toda la comunidad. Así, todo lo que viene de NumPy se escribe con el prefijo `np.`. **Hay que ejecutarla antes que cualquier otra celda.** La segunda línea muestra las versiones con las que se ejecuta el cuaderno: todo cuaderno del módulo las muestra.

In [2]:
import sys
import numpy as np   # Siempre hay que ejecutar esta celda antes que el resto

print("Python:", sys.version.split()[0])
print("NumPy: ", np.__version__)

Python: 3.14.8
NumPy:  2.5.4


# 1. Por qué existe el array

Python, por sí solo, es un lenguaje de propósito general: no está pensado para hacer cuentas con millones de números. **NumPy** aporta una estructura de datos, el **array** (`ndarray`), sobre la que se apoyan pandas, Matplotlib, scikit-learn y casi todas las librerías de datos.

## 1.1 Características del array

* **Homogéneo**: todos los elementos son del mismo tipo de dato.
* **Contiguo en memoria**: los números se guardan uno detrás de otro, sin referencias intermedias.
* **Tamaño fijo**: una vez creado, no crece ni encoge (a diferencia de las listas, que tienen `append`).
* **Multidimensional**: puede tener 1, 2, 3 o más dimensiones (vector, matriz, cubo…).
* **Vectorizado**: las operaciones se aplican a todos los elementos a la vez, sin escribir bucles.

## 1.2 La lista no sabe hacer cuentas

Una lista de Python es una caja de objetos cualesquiera. Multiplicarla por 2 **la repite**, y multiplicarla por un decimal da error.

In [3]:
importes_lista = [100.0, 250.0, 80.0]
print(importes_lista * 2)           # repite la lista

try:
    importes_lista * 1.21           # IVA del 21 %
except TypeError as e:
    print("TypeError:", e)

[100.0, 250.0, 80.0, 100.0, 250.0, 80.0]
TypeError: can't multiply sequence by non-int of type 'float'


El array, en cambio, opera **elemento a elemento**:

In [4]:
importes = np.array([100.0, 250.0, 80.0])
print(importes * 2)
print(importes * 1.21)

[200. 500. 160.]
[121.  302.5  96.8]


## 1.3 La diferencia de velocidad, medida

Vamos a calcular el precio final (IVA del 21 % y descuento del 10 %, redondeado a céntimos) de **2.000.000 de importes**: primero con un bucle de Python y después con NumPy.

Para comparar las velocidades, se usa `time.perf_counter()`, que devuelve el tiempo en segundos con la mayor precisión posible. Se mide el tiempo antes y después de cada cálculo y se resta.

In [10]:
import random, time

random.seed(1) # Para usar la misma lista de aleatorios cada vez
lista = [random.randint(1, 200) for _ in range(2_000_000)]
arr = np.array(lista)

# Medir el tiempo de cada cálculo usando listas
inicio = time.perf_counter()
resultado = []
for x in lista:
    resultado.append(round(x * 1.21, 2))
t_bucle = time.perf_counter() - inicio

# Medir el tiempo de cada cálculo usando NumPy
inicio = time.perf_counter()
resultado_np = np.round(arr * 1.21, 2)
t_numpy = time.perf_counter() - inicio

print(f"Bucle de Python: {t_bucle:.3f} s")
print(f"NumPy:           {t_numpy:.4f} s")
print(f"NumPy es unas {t_bucle / t_numpy:.0f} veces más rápido")
print("¿Mismo resultado?", np.allclose(resultado, resultado_np))

Bucle de Python: 0.680 s
NumPy:           0.0101 s
NumPy es unas 67 veces más rápido
¿Mismo resultado? True


**¿Por qué tanta diferencia?**

* En la lista, cada elemento es un objeto independiente. El bucle de Python tiene que comprobar el tipo de cada uno y llamar a la operación **una vez por elemento**.
* En el array, todos los números son del mismo tipo y están seguidos en memoria. NumPy hace la operación **una sola vez para todo el bloque**, en código compilado (C).
* Cuantos más datos, mayor es la diferencia.

> **La idea que hay que llevarse:** en análisis de datos **no se recorren las filas con bucles; se opera con la columna entera**. Es lo que hará pandas por debajo.

## 1.4 Lo que el array no sabe hacer

Un array solo tiene **un tipo de dato**. Si se mezclan números y textos, todo se convierte en texto:

In [5]:
mezcla = np.array([10, 20, "Hola", 40, True, 2.9])
print(mezcla)
print(mezcla.dtype)    # <U32: texto

['10' '20' 'Hola' '40' 'True' '2.9']
<U32


Una tabla real (nombre de cliente, provincia, fecha, importe) tiene columnas de tipos distintos y con nombre. El array no puede representarla bien: para eso existe **pandas**, que guarda cada columna como un array de su propio tipo.

# 2. Crear arrays

## 2.1 A partir de una lista o una tupla

Se utiliza la función `np.array()`, pasando como argumento una lista (o una tupla) de Python.

In [6]:
a = np.array([10, 20, 30, 40, 50])      # desde una lista
print(a)
b = np.array((10, 20, 30, 40, 50))      # desde una tupla
print(b)
m = np.array([[1, 2, 3],
              [4, 5, 6]])               # lista de listas: array de dos dimensiones
print(m)

[10 20 30 40 50]
[10 20 30 40 50]
[[1 2 3]
 [4 5 6]]


## 2.2 A partir de rangos de números

### `arange()`
Es un método similar a `range()` de Python, pero lo que hace es devuelver un array: `np.arange(inicio, fin, paso)`. El final **no se incluye**.

In [ ]:
print(np.arange(1, 10))        # Array con elementos del 1 al 9
print(np.arange(1, 10, 2))     # Ahi¡ora de de 2 en 2
print(np.arange(10, 1, -1))    # Ahora númeroos del 10 al 2
print(np.arange(0, 1, 0.25))   # Array con rango de números decimales

[1 2 3 4 5 6 7 8 9]
[1 3 5 7 9]
[10  9  8  7  6  5  4  3  2]
[0.   0.25 0.5  0.75]


### `linspace()`
Crea arrays donde la distancia entre cada elemento es la misma. Para ello se indica el inicio, el final (que **sí se incluye**) y **cuántos valores** se quieren, igualmente espaciados.

In [8]:
print(np.linspace(1, 10, 3))     # 3 valores entre 1 y 10
print(np.linspace(0, 1, 5))      # 5 valores entre 0 y 1

[ 1.   5.5 10. ]
[0.   0.25 0.5  0.75 1.  ]


## 2.3 Arrays rellenos de un mismo valor

* `np.zeros(forma)`: ceros.
* `np.ones(forma)`: unos.
* `np.full(forma, valor)`: cualquier valor.

La forma es un número (que indica un array de una dimensión) o una tupla (varias dimensiones).

In [9]:
print(np.zeros(5))
print(np.ones((2, 5)))          # 2 filas y 5 columnas
print(np.full((2, 3), 9))       # 2 filas y 3 columnas de nueves

[0. 0. 0. 0. 0.]
[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]
[[9 9 9]
 [9 9 9]]


Las variantes `zeros_like`, `ones_like` y `full_like` crean un array **con la misma forma y tipo que otro**:

In [10]:
plantilla = np.array([[1.5, 2.5], [3.5, 4.5]])
print(np.zeros_like(plantilla))

[[0. 0.]
 [0. 0.]]


`np.empty(forma)` crea un array **sin inicializar**: contiene lo que hubiera en la memoria. Es más rápido, pero solo tiene sentido si se van a rellenar todos los valores después.

In [11]:
e = np.empty(4)
print(e)      # valores impredecibles

[0. 0. 0. 0.]


## 2.4 Arrays con valores aleatorios

La forma recomendada es crear un **generador** con `np.random.default_rng(semilla)` y pedirle los números.

**La semilla** es el valor inicial del generador: con la misma semilla se obtienen **siempre los mismos números**. Es lo que hace que un análisis con datos aleatorios sea **reproducible**.

In [ ]:
rng = np.random.default_rng(42)          # generador con semilla 42

print(rng.integers(1, 10, 5))            # 5 enteros entre 1 y 9 (el final no se incluye)
print(rng.integers(1, 100, (3, 4)))      # matriz 3x4 de enteros entre 1 y 99
print(rng.random(5))                     # 5 decimales entre 0 y 1
print(rng.normal(0, 1, (2, 3)))          # Genera números en distribución normal de media 0 y desviación típica 1

[1 7 6 4 4]
[[86  9 70 20]
 [10 53 97 73]
 [76 72 78 51]]
[0.45038594 0.37079802 0.92676499 0.64386512 0.82276161]
[[ 0.46750934 -0.85929246  0.36875078]
 [-0.9588826   0.8784503  -0.04992591]]


In [13]:
# Con la misma semilla, los mismos números
print(np.random.default_rng(42).integers(1, 10, 5))
print(np.random.default_rng(42).integers(1, 10, 5))

[1 7 6 4 4]
[1 7 6 4 4]


Otros métodos del generador:

| Método | Qué genera |
|---|---|
| `rng.integers(bajo, alto, tamaño)` | Enteros entre `bajo` y `alto - 1` |
| `rng.random(tamaño)` | Decimales entre 0 y 1 |
| `rng.uniform(bajo, alto, tamaño)` | Decimales con distribución uniforme |
| `rng.normal(media, desviación, tamaño)` | Distribución normal |
| `rng.standard_normal(tamaño)` | Normal de media 0 y desviación 1 |
| `rng.binomial(n, p, tamaño)` | Binomial: éxitos en `n` ensayos de probabilidad `p` |
| `rng.poisson(lam, tamaño)` | Poisson de media `lam` |
| `rng.choice(a, tamaño, replace, p)` | Muestra aleatoria de `a`, con o sin reemplazo y con probabilidades `p` |
| `rng.permutation(x)` | Copia de `x` con los elementos en orden aleatorio |
| `rng.shuffle(x)` | Desordena `x` **en el sitio** |

`rng.choice` es la base del **muestreo aleatorio** que se hizo en Excel en la UT1:

In [14]:
rng = np.random.default_rng(7)
clientes = np.arange(1, 101)                            # 100 clientes
muestra = rng.choice(clientes, size=10, replace=False)  # muestra del 10 % sin repetir
print(muestra)

[58 85 75 81 55 31 86  6 23 64]


> **Nota: la forma antigua.** Muchos tutoriales usan funciones sueltas del módulo `np.random` con una semilla global: `np.random.seed(42)`, `np.random.randint(1, 10, 5)`, `np.random.random(5)`… Siguen funcionando, pero NumPy recomienda el generador: es más rápido, tiene mejores propiedades estadísticas y no depende de un estado compartido por todo el programa. Para que la reconozcas:

In [15]:
np.random.seed(42)                     # semilla global (forma antigua); no se asigna a nada
print(np.random.randint(1, 10, 5))     # equivalente antiguo de rng.integers
print(np.random.random(3))             # equivalente antiguo de rng.random

[7 4 8 5 7]
[0.44583275 0.09997492 0.45924889]


> **Ojo.** `np.random.seed(42)` no devuelve nada: escribir `a = np.random.seed(42)` deja `a` con el valor `None`.

# 3. Atributos y ejes

Todo array tiene unos atributos que describen su estructura:

| Atributo | Qué indica |
|---|---|
| `shape` | La **forma**: número de elementos en cada dimensión, como tupla |
| `ndim` | El número de dimensiones |
| `size` | El número total de elementos |
| `dtype` | El tipo de dato de los elementos |
| `itemsize` | Bytes que ocupa cada elemento |
| `nbytes` | Bytes que ocupa el array entero (`itemsize * size`) |

In [16]:
a = np.arange(1, 11)
b = np.array([[2, 3, 4], [5, 6, 7]])

print("a.shape:", a.shape, "  b.shape:", b.shape)   # (10,) y (2, 3)
print("a.ndim: ", a.ndim, "     b.ndim: ", b.ndim)  # 1 y 2
print("a.size: ", a.size, "    b.size: ", b.size)   # 10 y 6
print("b.dtype:", b.dtype)
print("b.itemsize:", b.itemsize, "bytes   b.nbytes:", b.nbytes, "bytes")

a.shape: (10,)   b.shape: (2, 3)
a.ndim:  1      b.ndim:  2
a.size:  10     b.size:  6
b.dtype: int64
b.itemsize: 8 bytes   b.nbytes: 48 bytes


> **Ojo con la forma de una dimensión.** `(10,)` es una tupla de un solo elemento: el array tiene 10 elementos en una dimensión. No es lo mismo que `(10, 1)` (10 filas y 1 columna) ni que `(1, 10)` (1 fila y 10 columnas).

## 3.1 Los ejes

En un array de dos dimensiones, cada dimensión es un **eje**:

```
                 eje 1  (columnas)  ──────────►
              col 0   col 1   col 2   col 3
            ┌───────┬───────┬───────┬───────┐
  eje 0  │  │  10   │  20   │  30   │  40   │  fila 0
(filas)  │  ├───────┼───────┼───────┼───────┤
         │  │  50   │  60   │  70   │  80   │  fila 1
         ▼  ├───────┼───────┼───────┼───────┤
            │  90   │ 100   │ 110   │ 120   │  fila 2
            └───────┴───────┴───────┴───────┘
                         shape = (3, 4)
```

* **Eje 0**: las filas. Es el primer número de la forma (3).
* **Eje 1**: las columnas. Es el segundo número de la forma (4).

En una tabla de datos, cada **fila** es un registro (un cliente, un ticket, una tienda) y cada **columna** es una variable. Esta idea es la que se usa en los apartados 11 y 12 para calcular totales por fila o por columna.

# 4. Tipos de dato

NumPy busca **velocidad y precisión**. Los tipos de Python (`int`, `float`) son demasiado genéricos: un `int` de Python puede crecer sin límite, y eso tiene un coste. NumPy usa tipos de **tamaño fijo**, como los de C, mucho más eficientes.

## 4.1 El atributo `dtype`

In [17]:
print(np.array([10, 20, 30]).dtype)          # int64
print(np.array([1.5, 2.0]).dtype)            # float64
print(np.array([1, 2.5, 3]).dtype)           # float64: un decimal convierte a todos
print(np.array([True, False]).dtype)         # bool
print(np.array(["León", "Soria"]).dtype)     # <U5: texto de hasta 5 caracteres

int64
float64
float64
bool
<U5


> Desde **NumPy 2.0**, el entero por defecto es **`int64`** en todos los sistemas, también en Windows de 64 bits (antes era `int32` en Windows).

Cuando se mezclan tipos, NumPy convierte todo al **tipo más general** que los pueda representar: `bool` → entero → decimal → texto.

## 4.2 Tabla de tipos numéricos

**Booleano y enteros**

| Tipo | Valores | Bytes | Rango |
|---|---|---|---|
| `bool` | Verdadero o falso | 1 | `True` o `False` |
| `int8` | Enteros pequeños | 1 | −128 a 127 |
| `int16` | Enteros | 2 | −32.768 a 32.767 |
| `int32` | Enteros | 4 | −2³¹ a 2³¹−1 (unos ±2.100 millones) |
| `int64` | Enteros (por defecto) | 8 | −2⁶³ a 2⁶³−1 |
| `uint8` | Enteros sin signo | 1 | 0 a 255 |
| `uint16` | Enteros sin signo | 2 | 0 a 65.535 |
| `uint32` | Enteros sin signo | 4 | 0 a 2³²−1 |
| `uint64` | Enteros sin signo | 8 | 0 a 2⁶⁴−1 |

**Decimales de coma flotante y complejos**

| Tipo | Valores | Bytes |
|---|---|---|
| `float16` | Decimal de media precisión | 2 |
| `float32` | Decimal de precisión simple (unos 7 dígitos) | 4 |
| `float64` | Decimal de precisión doble (unos 15-16 dígitos). **Por defecto** | 8 |
| `complex64` | Complejo formado por dos `float32` | 8 |
| `complex128` | Complejo formado por dos `float64`. Por defecto | 16 |

## 4.3 Crear un array de un tipo concreto: `dtype`

La mayoría de funciones de creación aceptan el parámetro `dtype`:

In [18]:
print(np.array([1, 2, 3], dtype="float64"))
print(np.zeros(3, dtype=np.int32))
print(np.ones(3, dtype=bool))

[1. 2. 3.]
[0 0 0]
[ True  True  True]


## 4.4 Convertir a otro tipo: `astype()`

`astype()` devuelve **una copia** del array con el tipo nuevo. Es la forma habitual de convertir, y la misma que se usará en pandas.

In [19]:
x = np.array([19.7, 25.2, 31.9])
print(x.astype(int))          # trunca los decimales (no redondea)
print(np.round(x).astype(int))  # para redondear: primero round

textos = np.array(["12", "7", "30"])
numeros = textos.astype(int)
print(numeros, numeros.dtype, numeros.sum())

print(np.float64(np.array([19, 25])))   # otra forma: el tipo como función

[19 25 31]
[20 25 32]
[12  7 30] int64 49
[19. 25.]


## 4.5 Desbordamiento

Un tipo de datos de tamaño fijo tiene un máximo. Si se supera, el valor pierde el sentido. Por ejemplo en un número entero es como si le diéramos la vuelta:

In [20]:
pequeno = np.array([127], dtype=np.int8)
print(pequeno + np.int8(1))     # -128: se ha desbordado

[-128]


**Precisión de los decimales.** Los decimales se guardan en binario y muchos no tienen representación exacta. Por eso **nunca se comparan decimales con `==`** (apartado 14):

In [21]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)            # False
print(np.isclose(0.1 + 0.2, 0.3))  # True: comparación con tolerancia

0.30000000000000004
False
True


# 5. Índices y rebanadas

## 5.1 Acceder y modificar un elemento

El primer elemento tiene índice **0**. Los índices **negativos** cuentan desde el final.

In [22]:
a = np.array([10, 20, 30, 40, 50])
print(a[2])      # 30
print(a[-1])     # 50: el último
print(a[-2])     # 40: el penúltimo

a[2] = 90        # los arrays son mutables
print(a)

30
50
40
[10 20 90 40 50]


En dos dimensiones, los índices de cada eje se separan **con una coma**: `[fila, columna]`.

In [23]:
m = np.array([[10, 20, 30, 40],
              [50, 60, 70, 80],
              [90, 100, 110, 120]])
print(m[1, 2])     # fila 1, columna 2: 70
print(m[-1, -1])   # última fila, última columna: 120

70
120


## 5.2 Rebanadas (*slicing*)

Igual que en las listas: `array[inicio:fin:paso]`. El **fin no se incluye**. Si se omite el inicio, empieza en 0; si se omite el fin, llega hasta el final; si se omite el paso, va de uno en uno.

In [24]:
a = np.arange(1, 11)
print("a          ->", a)
print("a[2:5]     ->", a[2:5])
print("a[:5]      ->", a[:5])
print("a[2:]      ->", a[2:])
print("a[2:7:2]   ->", a[2:7:2])
print("a[::-1]    ->", a[::-1])     # al revés
print("a[-3:]     ->", a[-3:])      # los tres últimos

a          -> [ 1  2  3  4  5  6  7  8  9 10]
a[2:5]     -> [3 4 5]
a[:5]      -> [1 2 3 4 5]
a[2:]      -> [ 3  4  5  6  7  8  9 10]
a[2:7:2]   -> [3 5 7]
a[::-1]    -> [10  9  8  7  6  5  4  3  2  1]
a[-3:]     -> [ 8  9 10]


## 5.3 Rebanadas en dos dimensiones

Se pone una rebanada por eje, separadas por coma. **`:` solo significa «todo ese eje»**.

In [25]:
b = np.arange(1, 13).reshape(4, 3)    # 4 filas y 3 columnas
print("b\n", b)
print("b[1]      (fila 1)            ->", b[1])
print("b[:, 1]   (columna 1)         ->", b[:, 1])
print("b[1, :2]  (fila 1, 2 primeras)->", b[1, :2])
print("b[:2, :2] (2 primeras filas y columnas)\n", b[:2, :2])

b
 [[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]
b[1]      (fila 1)            -> [4 5 6]
b[:, 1]   (columna 1)         -> [ 2  5  8 11]
b[1, :2]  (fila 1, 2 primeras)-> [4 5]
b[:2, :2] (2 primeras filas y columnas)
 [[1 2]
 [4 5]]


> **Ojo.** `b[:, 1]` (una columna) devuelve un array de **una dimensión**, forma `(4,)`, no una columna `(4, 1)`. Si se necesita conservar la forma de columna: `b[:, [1]]` o `b[:, 1:2]`.

# 6. Máscaras booleanas

Es la forma de **filtrar** datos: la herramienta que más se usa en análisis.

## 6.1 Comparar devuelve un array de booleanos

Una comparación entre un array y un valor se hace elemento a elemento, y devuelve un array de `True` y `False` con la misma forma. A ese array se le llama **máscara**.

In [26]:
a = np.arange(1, 11)
mascara = a > 5
print(mascara)

[False False False False False  True  True  True  True  True]


## 6.2 Filtrar con la máscara

Si se usa la máscara como índice, se obtienen **solo los elementos donde vale `True`**:

In [27]:
print(a[mascara])        # [ 6  7  8  9 10]
print(a[a % 2 == 0])     # se puede escribir la condición directamente: los pares

[ 6  7  8  9 10]
[ 2  4  6  8 10]


## 6.3 Operar mediante booleanas

Al sumar una máscara, cada `True` cuenta como 1. Es la forma de contar **cuántos elementos cumplen una condición**. EL método `mean` permite obtener la proporción de elementos que cumplen la condicion:

In [28]:
edades = np.array([23, 45, 67, 31, 52, 19, 74, 38])
print("Mayores de 50:", (edades > 50).sum())
print("Proporción de mayores de 50:", (edades > 50).mean())
print("Lo mismo con count_nonzero:", np.count_nonzero(edades > 50))

Mayores de 50: 3
Proporción de mayores de 50: 0.375
Lo mismo con count_nonzero: 3


## 6.4 Combinar condiciones: `&`, `|`, `~`

| Operador | Significado |
|---|---|
| `&` | y |
| `\|` | o |
| `~` | no |

**Cada condición va entre paréntesis.** Con arrays **no se usan** `and`, `or` ni `not`: dan error.

In [29]:
print(edades[(edades >= 30) & (edades < 60)])     # entre 30 y 59
print(edades[(edades < 25) | (edades > 70)])      # menores de 25 o mayores de 70
print(edades[~(edades > 50)])                     # los que NO son mayores de 50

try:
    edades[edades >= 30 and edades < 60]
except ValueError as e:
    print("ValueError:", e)

[45 31 52 38]
[23 19 74]
[23 45 31 19 38]
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


> **Ojo.** Sin paréntesis, `edades >= 30 & edades < 60` se evalúa en otro orden (el `&` tiene prioridad sobre las comparaciones) y da un error o un resultado incorrecto.

## 6.5 `any()` y `all()`

In [30]:
print("¿Alguno mayor de 70?", (edades > 70).any())
print("¿Todos mayores de 18?", (edades > 18).all())

¿Alguno mayor de 70? True
¿Todos mayores de 18? True


## 6.6 `np.where()`

Tiene dos usos:

* Con **un argumento**, devuelve las **posiciones** donde la condición es cierta.
* Con **tres argumentos**, `np.where(condición, si_cierto, si_falso)`, construye un array nuevo eligiendo valor elemento a elemento.

In [31]:
print(np.where(edades > 50))                  # tupla con el array de posiciones
print(np.where(edades > 50)[0])               # solo las posiciones
print(np.where(edades >= 65, "jubilado", "activo"))

(array([2, 4, 6]),)
[2 4 6]
['activo' 'activo' 'jubilado' 'activo' 'activo' 'activo' 'jubilado'
 'activo']


## 6.7 Modificar con una máscara

Asignar a una selección con máscara cambia **solo** esos elementos del array original. Es la forma de corregir valores imposibles, como los 999 que aparecían en la UT1 como edad desconocida:

In [32]:
edades = np.array([23, 999, 67, 31, 999, 19])
edades[edades == 999] = 0          # aquí los ponemos a 0 solo para ver el mecanismo
print(edades)

[23  0 67 31  0 19]


> En el apartado 13 se verá la forma correcta: convertirlos en **valores ausentes** (`np.nan`), porque un 999 no es una edad y un 0 tampoco.

# 7. Indexación con listas de índices

Se puede indexar con **una lista (o un array) de posiciones** para elegir elementos o filas concretos, en el orden que se quiera.

In [33]:
a = np.arange(1, 21).reshape(5, 4)
print("a\n", a)
print("Filas 1 y 3\n", a[[1, 3]])
print("Filas 4, 0 y 0 (se pueden repetir)\n", a[[4, 0, 0]])

v = np.array([10, 20, 30, 40, 50])
print("Elementos 0, 2 y 4:", v[[0, 2, 4]])

a
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]
 [17 18 19 20]]
Filas 1 y 3
 [[ 5  6  7  8]
 [13 14 15 16]]
Filas 4, 0 y 0 (se pueden repetir)
 [[17 18 19 20]
 [ 1  2  3  4]
 [ 1  2  3  4]]
Elementos 0, 2 y 4: [10 30 50]


Se usa mucho junto con `argsort` y `argmax` (apartados 11 y 18): primero se calculan las posiciones y después se seleccionan.

# 8. Vista frente a copia

Cuando se obtiene un array a partir de otro, NumPy puede devolver:

* una **vista**: un array nuevo que **comparte la memoria** con el original. **Si se modifica uno, cambia el otro**;
* una **copia**: un array independiente.

NumPy usa vistas siempre que puede porque no gasta memoria ni tiempo en copiar. El riesgo es modificar el original sin darse cuenta.

## 8.1 Qué devuelve cada operación

| Operación | Devuelve | ¿Cambiar el resultado cambia el original? |
|---|---|---|
| Rebanada: `a[2:5]`, `m[0]`, `m[:, 1]`, `m[:2, 1:]` | **Vista** | **Sí** |
| `reshape`, `ravel`, `.T` / `transpose` | **Vista** si es posible | **Sí**, normalmente |
| Máscara booleana: `a[a > 5]` | Copia | No |
| Lista de índices: `a[[0, 2]]` | Copia | No |
| `flatten()`, `astype()`, `.copy()` | Copia | No |
| Operaciones (`a * 2`, `np.sqrt(a)`…) | Array nuevo | No |

## 8.2 La rebanada es una vista

In [34]:
a = np.arange(1, 13).reshape(4, 3)
vista = a[:2, 1:3]          # dos primeras filas, dos últimas columnas
vista[0, 0] = 99            # modificamos la vista...
print(a)                    # ...y el original ha cambiado

[[ 1 99  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]


## 8.3 La máscara es una copia

In [35]:
a = np.arange(1, 13).reshape(4, 3)
grandes = a[a > 6]          # máscara: copia
grandes[:] = 0              # modificamos la copia...
print(a)                    # ...y el original no cambia

[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]


> **Ojo con la diferencia entre seleccionar y asignar.** `grandes = a[a > 6]` crea una copia; pero `a[a > 6] = 0` (asignar directamente sobre la selección) **sí modifica `a`**, como se vio en el apartado 6.7.

## 8.4 `reshape` y `transpose` también devuelven vistas

In [36]:
a = np.arange(6)
r = a.reshape(2, 3)
r[0, 0] = 100
print(a)            # el original también cambia: r es una vista

[100   1   2   3   4   5]


## 8.5 Comprobar si comparten memoria y hacer una copia

`np.shares_memory(x, y)` dice si dos arrays comparten datos. Para trabajar sin miedo a modificar el original: **`.copy()`**.

In [37]:
a = np.arange(1, 13).reshape(4, 3)
print("Rebanada:", np.shares_memory(a, a[:, 0]))
print("Máscara: ", np.shares_memory(a, a[a > 6]))
print("reshape: ", np.shares_memory(a, a.reshape(3, 4)))
print("flatten: ", np.shares_memory(a, a.flatten()))

copia = a[:2, 1:3].copy()
copia[0, 0] = 88
print(a[0, 1])      # sigue siendo 2

Rebanada: True
Máscara:  False
reshape:  True
flatten:  False
2


> **Para recordar: rebanada → vista; máscara o lista de índices → copia; si vas a modificar, `.copy()`.**
>
> En **pandas 3** todo se comporta como si fuera una copia (la *copia en escritura*), y eso cambia resultados respecto a versiones anteriores, como se vio en el segundo choque de la J5. En NumPy, en cambio, las vistas siguen siendo vistas.

# 9. Operaciones vectorizadas

**Vectorizar** es operar con arrays enteros en lugar de recorrer sus elementos con un bucle. Todas las operaciones de este apartado se aplican **elemento a elemento**.

## 9.1 Operaciones aritméticas entre arrays de la misma forma

In [38]:
a = np.arange(10, 70, 10).reshape(2, 3)
b = np.arange(5, 35, 5).reshape(2, 3)
print("a\n", a)
print("b\n", b)
print("a + b\n", a + b)
print("a - b\n", a - b)
print("a * b\n", a * b)
print("a / b\n", a / b)
print("a % b\n", a % b)
print("a ** 2\n", a ** 2)

a
 [[10 20 30]
 [40 50 60]]
b
 [[ 5 10 15]
 [20 25 30]]
a + b
 [[15 30 45]
 [60 75 90]]
a - b
 [[ 5 10 15]
 [20 25 30]]
a * b
 [[  50  200  450]
 [ 800 1250 1800]]
a / b
 [[2. 2. 2.]
 [2. 2. 2.]]
a % b
 [[0 0 0]
 [0 0 0]]
a ** 2
 [[ 100  400  900]
 [1600 2500 3600]]


> **Ojo.** `a * b` multiplica **elemento a elemento**, no es el producto de matrices del álgebra. El producto matricial se escribe `a @ b` (no se usa en este módulo).

## 9.2 Operaciones con un número (escalar)

El número se aplica a cada elemento:

In [39]:
a = np.arange(10, 70, 10).reshape(2, 3)
print("a + 3\n", a + 3)
print("a * 2\n", a * 2)
print("a > 25\n", a > 25)     # las comparaciones también son vectorizadas

a + 3
 [[13 23 33]
 [43 53 63]]
a * 2
 [[ 20  40  60]
 [ 80 100 120]]
a > 25
 [[False False  True]
 [ True  True  True]]


## 9.3 Asignación compuesta

`+=`, `-=`, `*=`, `/=` modifican el array **en el sitio**, sin crear uno nuevo:

In [40]:
a = np.arange(10, 70, 10).reshape(2, 3)
a += 3
print("Tras a += 3\n", a)
a *= 2
print("Tras a *= 2\n", a)

Tras a += 3
 [[13 23 33]
 [43 53 63]]
Tras a *= 2
 [[ 26  46  66]
 [ 86 106 126]]


> **Ojo.** `a /= 2` sobre un array de enteros da error, porque el resultado sería decimal y no cabe en el tipo del array. Se convierte antes: `a = a.astype(float)`.

## 9.4 Funciones universales

Las **funciones universales** (*ufuncs*) de NumPy se aplican elemento a elemento:

| Función | Qué hace |
|---|---|
| `np.sqrt`, `np.exp`, `np.log`, `np.log10` | Raíz, exponencial y logaritmos |
| `np.abs` | Valor absoluto |
| `np.round(x, decimales)` | Redondeo |
| `np.floor`, `np.ceil` | Redondeo hacia abajo y hacia arriba |
| `np.sin`, `np.cos`, `np.tan` | Trigonométricas |
| `np.maximum(a, b)`, `np.minimum(a, b)` | Máximo o mínimo elemento a elemento entre dos arrays |
| `np.clip(x, mín, máx)` | Recorta los valores a un rango |

In [41]:
a = np.arange(1, 13).reshape(3, 4)
print("Raíz cuadrada\n", np.round(np.sqrt(a), 3))
print("Logaritmo natural\n", np.round(np.log(a), 3))

importes = np.array([-15.0, 80.0, 1200.0, 45.5])
print("Valor absoluto:", np.abs(importes))
print("Recortados a [0, 1000]:", np.clip(importes, 0, 1000))

Raíz cuadrada
 [[1.    1.414 1.732 2.   ]
 [2.236 2.449 2.646 2.828]
 [3.    3.162 3.317 3.464]]
Logaritmo natural
 [[0.    0.693 1.099 1.386]
 [1.609 1.792 1.946 2.079]
 [2.197 2.303 2.398 2.485]]
Valor absoluto: [  15.    80.  1200.    45.5]
Recortados a [0, 1000]: [   0.    80.  1000.    45.5]


Las funciones logarítmicas son muy útiles con datos de **cola larga** (importes, rentas, tiempos), como se vio en la UT1: el logaritmo «comprime» los valores grandes.

# 10. Difusión (broadcasting)

La **difusión** es lo que permite operar arrays de **formas distintas**. Ya la has usado sin saberlo: `a + 3` suma un número a todos los elementos, como si el 3 se «estirara» hasta tener la forma de `a`.

## 10.1 Las reglas

Para operar dos arrays, NumPy compara sus formas **empezando por la derecha**, dimensión a dimensión. Dos dimensiones son **compatibles** si:

1. son **iguales**, o
2. una de ellas es **1**, o
3. una de ellas **no existe** (el array tiene menos dimensiones).

La dimensión que vale 1 (o que falta) se «estira» hasta la otra. Si alguna pareja no cumple las reglas, da **error**.

```
   (3, 4)          (3, 4)            (3, 4)            (3, 4)
 +    (4,)       +   (3, 1)        +    (3,)         + (1, 4)
 ─────────       ─────────         ─────────         ─────────
   (3, 4)  ✔       (3, 4)  ✔         ERROR ✖           (3, 4) ✔
 4 = 4           4 frente a 1      4 ≠ 3             4 = 4
 3 frente a —    3 = 3                               3 frente a 1
```

In [42]:
m = np.zeros((3, 4))
fila = np.array([1, 2, 3, 4])            # forma (4,)
columna = np.array([[10], [20], [30]])   # forma (3, 1)

print("m + fila      (3, 4) + (4,)\n", m + fila)          # la fila se repite en cada fila
print("m + columna   (3, 4) + (3, 1)\n", m + columna)     # la columna se repite en cada columna

m + fila      (3, 4) + (4,)
 [[1. 2. 3. 4.]
 [1. 2. 3. 4.]
 [1. 2. 3. 4.]]
m + columna   (3, 4) + (3, 1)
 [[10. 10. 10. 10.]
 [20. 20. 20. 20.]
 [30. 30. 30. 30.]]


## 10.2 El error de difusión, y cómo leerlo

In [43]:
try:
    m + np.array([10, 20, 30])          # forma (3,)
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


El mensaje da las dos formas: `(3,4)` y `(3,)`. Comparando por la derecha, **4 frente a 3**: no son iguales y ninguna es 1. La intención era sumar un valor por fila, así que el array de 3 valores tiene que ser una **columna** `(3, 1)`.

## 10.3 Convertir un vector en columna

Tres formas equivalentes:

In [44]:
v = np.array([10, 20, 30])                # forma (3,)
print(v[:, np.newaxis].shape)             # (3, 1): np.newaxis añade un eje
print(v.reshape(-1, 1).shape)             # (3, 1): el -1 se calcula solo
print((m + v[:, np.newaxis]))

(3, 1)
(3, 1)
[[10. 10. 10. 10.]
 [20. 20. 20. 20.]
 [30. 30. 30. 30.]]


Y una cuarta, que se ve en el apartado 12: calcular el total **con `keepdims=True`**, que ya lo deja con forma de columna.

## 10.4 Un uso típico: porcentajes por fila

Ventas de 3 tiendas en 4 trimestres. ¿Qué porcentaje de sus ventas anuales hace cada tienda en cada trimestre?

In [45]:
ventas = np.array([[100, 120, 90, 190],
                   [ 50,  60, 70,  20],
                   [ 80,  80, 80,  80]])
total_tienda = ventas.sum(axis=1)                  # forma (3,)
porcentaje = ventas / total_tienda[:, np.newaxis] * 100
print(np.round(porcentaje, 1))
print(porcentaje.sum(axis=1))                      # cada fila suma 100

[[20. 24. 18. 38.]
 [25. 30. 35. 10.]
 [25. 25. 25. 25.]]
[100. 100. 100.]


# 11. Agregación y estadísticos

Las funciones de **agregación** reducen muchos valores a uno: el total, la media, el máximo… Son las mismas que se calcularon en Excel en la UT1.

| Función | Qué calcula | Equivalente en Excel |
|---|---|---|
| `sum` | Suma | `SUMA` |
| `mean` | Media aritmética | `PROMEDIO` |
| `np.median` | Mediana | `MEDIANA` |
| `std` | Desviación típica | `DESVEST.P` (por defecto) o `DESVEST.M` (con `ddof=1`) |
| `var` | Varianza | `VAR.P` o `VAR.S` (con `ddof=1`) |
| `min`, `max` | Mínimo y máximo | `MIN`, `MAX` |
| `np.percentile(x, q)` | Percentil `q` (de 0 a 100) | `PERCENTIL.INC` |
| `np.quantile(x, q)` | Cuantil `q` (de 0 a 1) | `PERCENTIL.INC` |
| `argmin`, `argmax` | **Posición** del mínimo y del máximo | — |
| `cumsum` | Suma acumulada | — |
| `np.count_nonzero` | Cuántos elementos no son cero (o cuántos `True`) | `CONTAR.SI` |

Casi todas se pueden escribir como método (`a.sum()`) o como función (`np.sum(a)`). La mediana y los percentiles solo existen como función.

In [46]:
a = np.array([[10, 20, 30], [40, 50, 60], [70, 80, 90]])
print("Suma:      ", a.sum())
print("Media:     ", a.mean())
print("Mediana:   ", np.median(a))
print("Mínimo:    ", a.min(), "  Máximo:", a.max())
print("Percentil 25:", np.percentile(a, 25), "  Cuartiles:", np.percentile(a, [25, 50, 75]))
print("Posición del máximo:", a.argmax())

Suma:       450
Media:      50.0
Mediana:    50.0
Mínimo:     10   Máximo: 90


Percentil 25: 30.0   Cuartiles: [30. 50. 70.]
Posición del máximo: 8


## 11.1 Desviación típica: `ddof`

`np.std` calcula por defecto la desviación típica **poblacional** (divide entre *n*). En la UT1 usamos la **muestral** (`DESVEST.M`, divide entre *n − 1*), que es la que corresponde cuando los datos son una muestra. En NumPy se pide con **`ddof=1`**:

In [47]:
x = np.array([1, 2, 3, 4])
print("Poblacional (ddof=0):", x.std())
print("Muestral    (ddof=1):", x.std(ddof=1))   # = DESVEST.M de Excel

Poblacional (ddof=0): 1.118033988749895
Muestral    (ddof=1): 1.2909944487358056


> Con muchos datos la diferencia es despreciable; con pocos, no. **Para que tus resultados coincidan con los de Excel, usa `ddof=1`.**

## 11.2 Posiciones: `argmax` y `argmin`

Devuelven **dónde** está el máximo o el mínimo, no su valor. Se combinan con otro array para saber **quién** es:

In [48]:
tiendas = np.array(["León", "Burgos", "Soria", "Ávila"])
ventas = np.array([540, 610, 220, 305])
print("Tienda con más ventas:", tiendas[ventas.argmax()], ventas.max())
print("Tienda con menos:     ", tiendas[ventas.argmin()], ventas.min())

Tienda con más ventas: Burgos 610
Tienda con menos:      Soria 220


## 11.3 Suma acumulada

In [49]:
ventas_mes = np.array([100, 120, 90, 150])
print(ventas_mes.cumsum())        # ventas acumuladas a final de cada mes

[100 220 310 460]


# 12. Agregación por ejes

Con un array de dos dimensiones casi nunca se quiere un único total: se quiere **un total por fila** o **un total por columna**. Se indica con el parámetro **`axis`**.

## 12.1 La regla

> **El eje que indicas es el que desaparece.**

| Llamada | Qué hace | Forma resultante en un array (3, 4) |
|---|---|---|
| `a.sum()` | Suma todo | un número |
| `a.sum(axis=0)` | Recorre las filas **hacia abajo** y deja **un valor por columna** | `(4,)` |
| `a.sum(axis=1)` | Recorre las columnas **hacia la derecha** y deja **un valor por fila** | `(3,)` |

```
                 axis=1  ──────────►   (un valor por fila)
            ┌────┬────┬────┬────┐
 axis=0  │  │ 10 │ 20 │ 30 │ 40 │  → 100
   │     │  ├────┼────┼────┼────┤
   │     │  │ 50 │ 60 │ 70 │ 80 │  → 260
   ▼     ▼  ├────┼────┼────┼────┤
            │ 90 │100 │110 │120 │  → 420
            └────┴────┴────┴────┘
              ↓    ↓    ↓    ↓
             150  180  210  240     (un valor por columna)
```

In [50]:
a = np.arange(10, 130, 10).reshape(3, 4)
print(a)
print("axis=0 ->", a.sum(axis=0), " forma", a.sum(axis=0).shape)
print("axis=1 ->", a.sum(axis=1), " forma", a.sum(axis=1).shape)

[[ 10  20  30  40]
 [ 50  60  70  80]
 [ 90 100 110 120]]
axis=0 -> [150 180 210 240]  forma (4,)
axis=1 -> [100 260 420]  forma (3,)


Todas las funciones del apartado 11 admiten `axis`: `mean`, `std`, `min`, `max`, `argmax`, `np.median`, `np.percentile`…

## 12.2 `keepdims=True`

Conserva la dimensión que se agrega, con tamaño 1. Así el resultado **se puede combinar directamente** con el array original mediante difusión:

In [51]:
print(a.sum(axis=1).shape)                  # (3,)
print(a.sum(axis=1, keepdims=True).shape)   # (3, 1): lista para dividir por filas
print(np.round(a / a.sum(axis=1, keepdims=True), 3))

(3,)
(3, 1)
[[0.1   0.2   0.3   0.4  ]
 [0.192 0.231 0.269 0.308]
 [0.214 0.238 0.262 0.286]]


## 12.3 Con datos reales: ventas de 14 tiendas en 12 meses

El fichero `ventas_tienda_mes.csv` contiene la facturación de 2025 de las 14 tiendas de la cadena que usamos en la J5, **en miles de euros**: una fila por tienda y una columna por mes. Lo cargamos con `np.loadtxt` (se explica en el apartado 15). Hay que leerlo dos veces porque **un array solo tiene un tipo**: una para los números y otra para los códigos de tienda, que son texto.

In [52]:
ventas = np.loadtxt("ventas_tienda_mes.csv", delimiter=",", skiprows=1, usecols=range(1, 13))
tiendas = np.loadtxt("ventas_tienda_mes.csv", delimiter=",", skiprows=1, usecols=0, dtype=str)
meses = np.array(["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"])

print(ventas.shape, ventas.dtype)
print(tiendas)
print(ventas[:3])        # las tres primeras tiendas

(14, 12) float64
['T01' 'T02' 'T03' 'T04' 'T05' 'T06' 'T07' 'T08' 'T09' 'T10' 'T11' 'T12'
 'T13' 'T14']
[[530.8 456.9 564.1 585.1 622.3 547.  532.1 463.5 568.  626.3 704.2 969.6]
 [421.2 383.8 466.5 490.8 507.2 461.5 455.2 380.1 485.4 535.  584.5 816.3]
 [339.2 308.2 375.4 385.4 409.5 379.1 357.8 307.  402.3 420.1 463.7 644.4]]


In [53]:
print("Total de la cadena (miles de €):", ventas.sum().round(1))

total_mes = ventas.sum(axis=0)       # un valor por columna: por mes
total_tienda = ventas.sum(axis=1)    # un valor por fila: por tienda
print("Por mes:   ", total_mes.round(1))
print("Por tienda:", total_tienda.round(1))

print("Mes con más ventas:   ", meses[total_mes.argmax()])
print("Tienda que más factura:", tiendas[total_tienda.argmax()], total_tienda.max().round(1))

Total de la cadena (miles de €): 63274.7
Por mes:    [ 3746.4  3347.4  4081.1  4252.7  4472.7  4046.8  3957.8  3393.5  4278.2
  7610.2  8414.7 11673.2]
Por tienda: [ 7169.9  5987.5  4792.1  5006.1  3202.5 10854.8  5571.8  2403.1  5952.7
  2964.3  2619.6  2793.   2180.8  1776.5]
Mes con más ventas:    dic
Tienda que más factura: T06 10854.8


**Cuota de cada tienda en cada mes** (qué parte de las ventas del mes hace cada tienda): se divide cada columna por el total de su mes. `(14, 12) / (12,)`: la difusión funciona sin hacer nada.

In [54]:
cuota = ventas / total_mes
print("Cuota de T06 en diciembre:", cuota[5, 11].round(3))
print("Cada mes suma 1:", np.allclose(cuota.sum(axis=0), 1))

Cuota de T06 en diciembre: 0.394
Cada mes suma 1: True


**Reparto de cada tienda a lo largo del año** (qué parte de sus ventas anuales hace cada tienda en cada mes): hay que dividir cada fila por el total de su tienda. `(14, 12) / (14,)` **no** es compatible; hace falta la forma `(14, 1)`.

In [55]:
try:
    ventas / total_tienda
except ValueError as e:
    print("ValueError:", e)

reparto = ventas / ventas.sum(axis=1, keepdims=True)
print("Parte de las ventas anuales de T01 que hace en diciembre:", reparto[0, 11].round(3))

ValueError: operands could not be broadcast together with shapes (14,12) (14,) 
Parte de las ventas anuales de T01 que hace en diciembre: 0.135


# 13. Valores ausentes: NaN

## 13.1 Qué es `NaN`

`np.nan` (*Not a Number*, «no es un número») es el valor que usa NumPy para representar un **dato ausente**. Tres cosas que hay que saber:

1. Es un número **decimal**: solo puede estar en arrays `float`. Un array de enteros no admite huecos.
2. **Se propaga**: cualquier operación con un `NaN` da `NaN`.
3. **No es igual a nada, ni a sí mismo**: `np.nan == np.nan` es `False`.

In [56]:
x = np.array([2.0, np.nan, 4.0])
print(x.dtype)
print("Suma:", x.sum(), "  Media:", x.mean())      # nan: se propaga
print("nan == nan:", np.nan == np.nan)             # False

float64
Suma: nan   Media: nan
nan == nan: False


## 13.2 Detectar `NaN`: `np.isnan`

Como `NaN` no es igual a nada, **nunca se busca con `==`**: se usa `np.isnan()`, que devuelve una máscara.

In [57]:
print(x == np.nan)            # siempre False: no sirve
print(np.isnan(x))            # así sí
print("Ausentes:", np.isnan(x).sum())
print("Valores presentes:", x[~np.isnan(x)])

[False False False]
[False  True False]
Ausentes: 1
Valores presentes: [2. 4.]


## 13.3 Calcular ignorando los `NaN`: las funciones `nan…`

| Normal | Ignorando `NaN` |
|---|---|
| `np.sum` | `np.nansum` |
| `np.mean` | `np.nanmean` |
| `np.median` | `np.nanmedian` |
| `np.std`, `np.var` | `np.nanstd`, `np.nanvar` |
| `np.min`, `np.max` | `np.nanmin`, `np.nanmax` |
| `np.percentile` | `np.nanpercentile` |
| `np.argmin`, `np.argmax` | `np.nanargmin`, `np.nanargmax` |

In [58]:
print("nanmean:", np.nanmean(x))     # 3.0
print("nansum: ", np.nansum(x))      # 6.0

nanmean: 3.0
nansum:  6.0


También funcionan por ejes. Contar los ausentes **por columna** es una de las primeras cosas que se hace con un conjunto de datos (en Excel era `CONTAR.BLANCO`):

In [59]:
datos = np.array([[34.0, 120.5, 3.0],
                  [np.nan, 80.0, 1.0],
                  [41.0, np.nan, np.nan],
                  [29.0, 45.2, 2.0]])
print("Ausentes por columna:", np.isnan(datos).sum(axis=0))
print("Media por columna:   ", np.nanmean(datos, axis=0).round(2))

Ausentes por columna: [1 1 1]
Media por columna:    [34.67 81.9   2.  ]


## 13.4 Ausentes disfrazados

En los datos reales, la ausencia muchas veces no aparece como hueco, sino **disfrazada** de valor: un 999 en la edad, un 0 en el teléfono, un −1 en una puntuación. Si no se convierten en `NaN` **antes de calcular**, falsean los estadísticos. Se convierten con una máscara (el array tiene que ser `float`):

In [60]:
edades = np.array([23, 999, 67, 31, 999, 19]).astype(float)
print("Media con los 999:", edades.mean().round(1))
edades[edades == 999] = np.nan
print("Media sin ellos:  ", np.nanmean(edades).round(1))

Media con los 999: 356.3
Media sin ellos:   35.0


**Un caso real: la tienda T06.** En la matriz de ventas, la tienda T06 tiene **cero** ventas de enero a septiembre. Pero no es que no vendiera: **abrió el 1 de octubre**. Esos ceros son meses sin tienda, es decir, **ausentes**.

In [61]:
print(ventas[5])                                   # T06

t06 = ventas[5].copy()                             # copia: no tocamos la matriz original
t06[:9] = np.nan                                   # enero a septiembre: no existía
print("Media mensual de T06 con los ceros:", ventas[5].mean().round(1))
print("Media mensual de T06 sin ellos:    ", np.nanmean(t06).round(1))

[   0.     0.     0.     0.     0.     0.     0.     0.     0.  2976.3
 3279.8 4598.7]
Media mensual de T06 con los ceros: 904.6
Media mensual de T06 sin ellos:     3618.3


Con los ceros, T06 parece una tienda pequeña; sin ellos, es con diferencia la que más vende al mes. **Decidir si un cero es un cero o un ausente cambia la conclusión**, y esa decisión no la toma NumPy: la toma quien analiza, sabiendo qué significan los datos.

# 14. Verificar los resultados

Un análisis encadena muchos pasos. Si uno sale mal y no se detecta, el error llega hasta la conclusión. La costumbre que se pide en el módulo es **comprobar cada resultado intermedio**.

## 14.1 `assert`

`assert condición, "mensaje"` no hace nada si la condición es cierta; si es falsa, **detiene la ejecución** con un `AssertionError` y el mensaje.

In [62]:
assert ventas.shape == (14, 12), "La matriz no tiene 14 tiendas y 12 meses"
assert ventas.dtype == np.float64
print("Comprobaciones superadas")

Comprobaciones superadas


In [63]:
edades = np.array([23, 45, 999, 31])
try:
    assert ((edades >= 0) & (edades <= 120)).all(), "Hay edades fuera de rango"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: Hay edades fuera de rango


**Qué se comprueba después de cada paso:**

| Qué | Ejemplo |
|---|---|
| Forma | `assert x.shape == (14, 12)` |
| Tipo | `assert x.dtype == np.float64` |
| Rango | `assert ((x >= 0) & (x <= 120)).all()` |
| Sin ausentes | `assert not np.isnan(x).any()` |
| Totales que cuadran | `assert np.isclose(x.sum(axis=1).sum(), x.sum())` |
| Recuentos | `assert (x > 0).sum() == 168` |

## 14.2 Comparar decimales: `np.isclose` y `np.allclose`

Como se vio en el apartado 4.5, los decimales no se comparan con `==`:

* `np.isclose(a, b)`: compara con una pequeña tolerancia, elemento a elemento.
* `np.allclose(a, b)`: `True` si **todos** los elementos están cerca.

In [64]:
total_por_filas = ventas.sum(axis=1).sum()
total_por_columnas = ventas.sum(axis=0).sum()
print(total_por_filas == total_por_columnas)          # puede fallar por redondeo
assert np.isclose(total_por_filas, total_por_columnas)
assert np.allclose(cuota.sum(axis=0), 1)
print("Los totales cuadran")

True
Los totales cuadran


## 14.3 Leer los mensajes de error

Los tres errores más frecuentes al trabajar con arrays:

| Error | Ejemplo de mensaje | Qué significa |
|---|---|---|
| `TypeError` | `can't multiply sequence by non-int of type 'float'` | Estás operando con una **lista**, no con un array |
| `ValueError` | `operands could not be broadcast together with shapes (14,12) (14,)` | Formas incompatibles: compáralas por la derecha (apartado 10) |
| `IndexError` | `index 12 is out of bounds for axis 1 with size 12` | Índice fuera de rango: los índices van de 0 a tamaño − 1 |

In [65]:
try:
    ventas[0, 12]
except IndexError as e:
    print("IndexError:", e)

IndexError: index 12 is out of bounds for axis 1 with size 12


# 15. Leer y guardar datos con NumPy

NumPy puede leer ficheros de texto sencillos con números. Para ficheros reales —con columnas de texto, fechas, codificaciones o separadores raros— se usa **pandas**, pero conviene conocer estas funciones y ver dónde se quedan cortas.

Primero creamos un pequeño fichero de ejemplo, con huecos y un 999:

In [66]:
with open("ejemplo_numpy.csv", "w", encoding="utf-8") as f:
    f.write("edad,importe,visitas\n34,120.5,3\n,80.0,1\n999,45.2,\n41,,2\n")
print(open("ejemplo_numpy.csv", encoding="utf-8").read())

edad,importe,visitas
34,120.5,3
,80.0,1
999,45.2,
41,,2



## 15.1 `np.loadtxt`: ficheros completos

Lee un fichero de números **sin huecos**. Parámetros principales: `delimiter` (separador), `skiprows` (filas a saltar, normalmente la cabecera), `usecols` (qué columnas) y `dtype`. Con un hueco, falla:

In [67]:
try:
    np.loadtxt("ejemplo_numpy.csv", delimiter=",", skiprows=1)
except ValueError as e:
    print("ValueError:", e)

ValueError: could not convert string '' to float64 at row 1, column 1.


## 15.2 `np.genfromtxt`: ficheros con huecos

Igual que `loadtxt`, pero convierte los huecos en `NaN`. Los nombres de las columnas se leen aparte.

In [68]:
datos = np.genfromtxt("ejemplo_numpy.csv", delimiter=",", skip_header=1)
with open("ejemplo_numpy.csv", encoding="utf-8") as f:
    columnas = f.readline().strip().split(",")

print(columnas)
print(datos)
print("Ausentes por columna:", np.isnan(datos).sum(axis=0))

datos[datos == 999] = np.nan
print("Medias por columna:  ", np.nanmean(datos, axis=0).round(2))

['edad', 'importe', 'visitas']
[[ 34.  120.5   3. ]
 [  nan  80.    1. ]
 [999.   45.2   nan]
 [ 41.    nan   2. ]]
Ausentes por columna: [1 1 1]
Medias por columna:   [37.5 81.9  2. ]


## 15.3 Guardar: `np.savetxt` y el formato `.npy`

* `np.savetxt(fichero, array, delimiter=",", fmt="%.2f")`: guarda en texto (CSV).
* `np.save(fichero, array)` y `np.load(fichero)`: formato binario propio de NumPy (`.npy`). Conserva el tipo y la forma exactos y es mucho más rápido. Solo lo lee NumPy.

In [69]:
np.savetxt("medias.csv", np.nanmean(datos, axis=0).reshape(1, -1), delimiter=",", fmt="%.2f",
           header=",".join(columnas), comments="")
print(open("medias.csv", encoding="utf-8").read())

np.save("ventas.npy", ventas)
recuperado = np.load("ventas.npy")
assert np.array_equal(recuperado, ventas) and recuperado.dtype == ventas.dtype
print("Guardado y recuperado sin cambios:", recuperado.shape)

edad,importe,visitas
37.50,81.90,2.00

Guardado y recuperado sin cambios: (14, 12)


## 15.4 El límite de NumPy

* Una columna de texto (provincia, nombre) no cabe en un array numérico: hay que leerla aparte.
* No hay **nombres de columna**: hay que recordar que la columna 1 era el importe.
* No hay **tipos distintos por columna**, ni fechas, ni manejo de codificaciones o separadores decimales.

Todo eso lo resuelve **pandas**: un DataFrame es, por dentro, un conjunto de arrays de NumPy (uno por columna) con nombres y con un índice.

# 16. Cambiar la forma

## 16.1 `reshape`

Cambia la forma **sin cambiar los datos**. La forma nueva tiene que tener el mismo número de elementos. Un **`-1`** en una dimensión significa «calcúlala tú». Devuelve una **vista** siempre que puede (apartado 8).

In [70]:
a = np.arange(1, 10)
print(a.reshape(3, 3))
print(a.reshape(-1, 1).shape)     # columna: (9, 1)
print(np.arange(12).reshape(2, -1))  # 2 filas; las columnas se calculan: 6

try:
    a.reshape(2, 5)
except ValueError as e:
    print("ValueError:", e)

[[1 2 3]
 [4 5 6]
 [7 8 9]]
(9, 1)
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
ValueError: cannot reshape array of size 9 into shape (2,5)


## 16.2 Aplanar: `flatten` y `ravel`

Las dos convierten un array de varias dimensiones en uno de una sola:

* **`flatten()`** devuelve siempre una **copia**.
* **`ravel()`** devuelve una **vista** cuando puede (es más rápida, pero modificar el resultado puede modificar el original).

In [71]:
b = np.array([[2, 3, 4], [5, 6, 7]])
plano = b.flatten()
plano[0] = 100
print("Tras modificar flatten:", b[0, 0])     # 2: no cambia

plano = b.ravel()
plano[0] = 100
print("Tras modificar ravel:  ", b[0, 0])     # 100: era una vista

Tras modificar flatten: 2
Tras modificar ravel:   100


## 16.3 Transponer: `transpose` o `.T`

Intercambia filas por columnas. Devuelve una **vista**.

In [72]:
b = np.array([[2, 3, 4], [5, 6, 7]])
print(b.T)
print(b.T.shape)

[[2 5]
 [3 6]
 [4 7]]
(3, 2)


## 16.4 Añadir un eje: `np.newaxis`

Convierte un vector en fila o en columna. Es lo que se usa para la difusión (apartado 10).

In [73]:
v = np.array([1, 2, 3])
print(v.shape, v[np.newaxis, :].shape, v[:, np.newaxis].shape)

(3,) (1, 3) (3, 1)


# 17. Combinar y dividir arrays

## 17.1 `np.concatenate`

Une dos o más arrays a lo largo de un eje (por defecto, el 0). Las demás dimensiones tienen que coincidir.

In [74]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(np.concatenate((a, b)))

m1 = np.arange(1, 7).reshape(2, 3)
m2 = np.arange(7, 10).reshape(1, 3)
print(np.concatenate((m1, m2)))                       # una fila más (axis=0)
print(np.concatenate((m1, np.array([[10], [20]])), axis=1))   # una columna más

[1 2 3 4 5 6]
[[1 2 3]
 [4 5 6]
 [7 8 9]]
[[ 1  2  3 10]
 [ 4  5  6 20]]


## 17.2 `vstack` y `hstack`

* **`vstack`** apila en vertical (una debajo de otra). Admite vectores de una dimensión como filas.
* **`hstack`** apila en horizontal (una al lado de otra).

In [75]:
m = np.arange(1, 7).reshape(2, 3)
print(np.vstack([m, np.array([10, 20, 30])]))
print(np.hstack([m, np.array([[10], [20]])]))

[[ 1  2  3]
 [ 4  5  6]
 [10 20 30]]
[[ 1  2  3 10]
 [ 4  5  6 20]]


## 17.3 Dividir: `split`, `vsplit` y `hsplit`

`np.split(a, [i, j])` corta el array en las posiciones `i` y `j`. Con el desempaquetado se puede asignar cada trozo a una variable.

In [76]:
a = np.arange(1, 10)
print(np.split(a, [3, 5]))
x, y, z = np.split(a, [3, 5])
print("x ->", x, " y ->", y, " z ->", z)

[array([1, 2, 3]), array([4, 5]), array([6, 7, 8, 9])]
x -> [1 2 3]  y -> [4 5]  z -> [6 7 8 9]


`vsplit` divide por filas y `hsplit` por columnas:

In [77]:
a = np.arange(1, 13).reshape(3, 4)
arriba, abajo = np.vsplit(a, [2])        # las dos primeras filas y el resto
print("arriba\n", arriba)
print("abajo\n", abajo)
izquierda, derecha = np.hsplit(a, [2])   # las dos primeras columnas y el resto
print("izquierda\n", izquierda)

arriba
 [[1 2 3 4]
 [5 6 7 8]]
abajo
 [[ 9 10 11 12]]
izquierda
 [[ 1  2]
 [ 5  6]
 [ 9 10]]


# 18. Ordenar, valores únicos y conjuntos

## 18.1 `sort`: método frente a función

* **`a.sort()`** (método) ordena **el propio array**, en el sitio, y no devuelve nada.
* **`np.sort(a)`** (función) devuelve una **copia ordenada** y deja `a` como estaba.

In [78]:
a = np.array([4, 8, 4, 7, 2, 5, 1])
ordenado = np.sort(a)
print("np.sort(a):", ordenado, "  a sigue igual:", a)

a.sort()
print("Tras a.sort():", a)
print("Lo que devuelve a.sort():", a.sort())     # None

np.sort(a): [1 2 4 4 5 7 8]   a sigue igual: [4 8 4 7 2 5 1]
Tras a.sort(): [1 2 4 4 5 7 8]
Lo que devuelve a.sort(): None


Para ordenar **de mayor a menor**, se ordena y se invierte con `[::-1]`. Con `axis` se ordena cada columna o cada fila **por separado**:

In [79]:
print(np.sort(np.array([4, 8, 4, 7]))[::-1])

m = np.array([[30, 60, 10], [50, 20, 40]])
print("Cada columna ordenada (axis=0)\n", np.sort(m, axis=0))
print("Cada fila ordenada (axis=1)\n", np.sort(m, axis=1))

[8 7 4 4]
Cada columna ordenada (axis=0)
 [[30 20 10]
 [50 60 40]]
Cada fila ordenada (axis=1)
 [[10 30 60]
 [20 40 50]]


> **Ojo.** Ordenar cada columna por separado **rompe las filas**: los valores de una misma fila dejan de corresponder al mismo registro. Para ordenar los registros de una tabla por una columna se usa `argsort` (o pandas).

## 18.2 `argsort`: las posiciones que ordenan

Devuelve las posiciones que ordenarían el array. Sirve para ordenar **otro** array con el mismo criterio y para sacar **los *k* mayores**:

In [80]:
tiendas_ej = np.array(["León", "Burgos", "Soria", "Ávila", "Zamora"])
ventas_ej = np.array([540, 610, 220, 305, 410])

orden = ventas_ej.argsort()[::-1]          # de mayor a menor
print(orden)
print("Ranking:", tiendas_ej[orden])
print("Las 3 que más venden:", tiendas_ej[orden[:3]], ventas_ej[orden[:3]])

[1 0 4 3 2]
Ranking: ['Burgos' 'León' 'Zamora' 'Ávila' 'Soria']
Las 3 que más venden: ['Burgos' 'León' 'Zamora'] [610 540 410]


## 18.3 Valores únicos: `np.unique`

Devuelve los valores distintos, **ordenados**. Con `return_counts=True`, además, cuántas veces aparece cada uno: es una tabla de frecuencias.

In [81]:
provincias = np.array(["León", "Soria", "León", "Ávila", "León", "Soria"])
valores, cuentas = np.unique(provincias, return_counts=True)
print(valores)
print(cuentas)

['León' 'Soria' 'Ávila']
[3 2 1]


## 18.4 Operaciones de conjuntos

| Función | Resultado |
|---|---|
| `np.union1d(a, b)` | Elementos que están en `a` o en `b` |
| `np.intersect1d(a, b)` | Elementos que están en los dos |
| `np.setdiff1d(a, b)` | Elementos de `a` que no están en `b` |
| `np.setxor1d(a, b)` | Elementos que están en uno de los dos, pero no en ambos |
| `np.isin(a, b)` | Máscara: qué elementos de `a` están en `b` |

Todas eliminan los duplicados y ordenan el resultado.

In [82]:
a = np.array([10, 20, 30, 40, 50])
b = np.array([30, 40, 50, 60, 70])
print("Unión:        ", np.union1d(a, b))
print("Intersección: ", np.intersect1d(a, b))
print("Diferencia:   ", np.setdiff1d(a, b))
print("Dif. simétrica:", np.setxor1d(a, b))
print("isin:         ", np.isin(a, b))

Unión:         [10 20 30 40 50 60 70]
Intersección:  [30 40 50]
Diferencia:    [10 20]
Dif. simétrica: [10 20 60 70]
isin:          [False False  True  True  True]


> **Ojo con dos dimensiones.** Estas funciones **aplanan** los arrays y trabajan con elementos sueltos, no con filas:

In [83]:
a2 = np.array([[1, 2], [3, 4], [5, 6]])
b2 = np.array([[3, 4], [5, 6], [7, 8]])
print(np.intersect1d(a2, b2))     # [3 4 5 6]: elementos, no filas

[3 4 5 6]


# 20. Errores frecuentes

| Lo que ves | Qué pasa | Solución |
|---|---|---|
| `TypeError: can't multiply sequence by non-int of type 'float'` | Estás operando con una lista | Conviértela: `np.array(lista)` |
| Una lista multiplicada por 2 sale repetida | Lo mismo | `np.array(lista) * 2` |
| `ValueError: operands could not be broadcast together with shapes …` | Formas incompatibles | Compara las formas por la derecha; usa `[:, np.newaxis]` o `keepdims=True` |
| `ValueError: The truth value of an array … is ambiguous` | Has usado `and`, `or` o `if` con un array | `&`, `\|`, `~` con paréntesis; o `.any()` / `.all()` |
| `IndexError: index … is out of bounds` | Índice fuera de rango | Los índices van de 0 a tamaño − 1 |
| La media (o la suma) sale `nan` | Hay algún `NaN` | `np.nanmean`, `np.nansum`… |
| `x == np.nan` no encuentra nada | `NaN` no es igual a nada | `np.isnan(x)` |
| Modificas un resultado y cambia el original | Era una vista | `.copy()` |
| Modificas un resultado y el original no cambia, y querías cambiarlo | Era una copia (máscara) | Asigna directamente: `a[a > 5] = 0` |
| La desviación típica no coincide con Excel | `ddof` | `ddof=1` para `DESVEST.M` |
| `a.sort()` devuelve `None` | El método ordena en el sitio | `np.sort(a)` si quieres un array nuevo |
| `a = np.random.seed(42)` deja `a` vacío | `seed` no devuelve nada | Mejor: `rng = np.random.default_rng(42)` |
| Un valor «imposible» al sumar enteros pequeños | Desbordamiento del tipo | Usa `int64` o `float64` |
| `0.1 + 0.2 == 0.3` es `False` | Precisión de los decimales | `np.isclose` |

# 21. Chuleta

```python
import numpy as np

# Crear
np.array([1, 2, 3])               np.arange(0, 10, 2)          np.linspace(0, 1, 5)
np.zeros((3, 4))                  np.ones(5)                   np.full((2, 3), 7)
rng = np.random.default_rng(42)   rng.integers(1, 7, 10)       rng.normal(0, 1, (3, 3))

# Describir
a.shape   a.ndim   a.size   a.dtype   a.astype(float)

# Seleccionar
a[0]   a[-1]   a[2:5]   m[1, 2]   m[:, 0]   m[:2, 1:]
a[a > 5]   a[(a > 2) & (a < 8)]   a[[0, 3]]   np.where(a > 5)

# Operar (sin bucles)
a * 1.21   a + b   np.sqrt(a)   np.round(a, 2)   np.clip(a, 0, 100)

# Difusión: formas comparadas por la derecha
m / m.sum(axis=0)                  # (n, k) / (k,)
m / m.sum(axis=1, keepdims=True)   # (n, k) / (n, 1)

# Agregar: el eje que indicas desaparece
a.sum()   m.sum(axis=0)   m.mean(axis=1)   np.median(m, axis=0)
a.std(ddof=1)   np.percentile(a, [25, 50, 75])   a.argmax()   a.cumsum()

# Ausentes
np.isnan(a)   np.nanmean(a)   a[a == 999] = np.nan

# Verificar
assert m.shape == (14, 12)
assert ((a >= 0) & (a <= 120)).all(), "Fuera de rango"
assert np.isclose(x, y)

# Vista o copia
b = a[2:5]          # vista
b = a[a > 5]        # copia
b = a.copy()        # copia explícita

# Leer y guardar
np.loadtxt("f.csv", delimiter=",", skiprows=1)
np.genfromtxt("f.csv", delimiter=",", skip_header=1)    # huecos -> nan
np.save("f.npy", a)   np.load("f.npy")
```